# HouseAI: Intelligent House Price Prediction & Explainable AI Pipeline
### College AIML Project - Comprehensive Model Analysis & Benchmarking

This notebook provides the complete analytical workflow for the **HouseAI** system:
1. **Exploratory Data Analysis (EDA)** & distribution analysis
2. **Data Preprocessing & Feature Engineering**
3. **Multi-Model Training & Benchmarking** (Linear Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost)
4. **Evaluation Metrics** (MAE, MSE, RMSE, R² Score, MAPE)
5. **Model Explainability with SHAP** (Feature Importance & Attributions)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

# Load dataset
df = pd.read_csv('../dataset/house_prices.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 1. Exploratory Data Analysis & Statistics

In [ ]:
# Summary statistics
print("--- Summary Statistics ---")
display(df[['area', 'carpet_area', 'bhk', 'bathrooms', 'floor', 'property_age', 'price']].describe())

In [ ]:
# Distribution of House Prices by City
plt.figure(figsize=(12, 6))
df['price_lakhs'] = df['price'] / 100000
sns.boxplot(data=df, x='city', y='price_lakhs', palette='viridis')
plt.title('Property Valuation Distribution by City (in Lakhs INR)', fontsize=14, fontweight='bold')
plt.xlabel('City', fontsize=12)
plt.ylabel('Price (₹ Lakhs)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Feature Correlation Heatmap

In [ ]:
# Numerical correlation matrix
num_cols = ['area', 'carpet_area', 'bhk', 'bathrooms', 'balconies', 'floor', 'property_age', 'parking', 'amenities_count', 'price']
corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix with Property Price', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Model Benchmark Results
Comparing the 5 evaluated algorithms on the test partition (20% holdout):

In [ ]:
# Load genuine model benchmark results from training pipeline
with open('../backend/model/model_comparison.json', 'r') as f:
    comparison_data = json.load(f)

comp_df = pd.DataFrame(comparison_data)
comp_df['RMSE (₹ L)'] = comp_df['rmse'] / 100000
comp_df['MAE (₹ L)'] = comp_df['mae'] / 100000
display(comp_df[['model_name', 'r2_score', 'train_r2_score', 'RMSE (₹ L)', 'MAE (₹ L)', 'mape_percent']])

# Bar chart comparison of R2 Score
plt.figure(figsize=(10, 5))
sns.barplot(data=comp_df, x='r2_score', y='model_name', palette='crest')
plt.title('R² Validation Score Comparison across Models', fontsize=14, fontweight='bold')
plt.xlabel('R² Score (Higher is Better)', fontsize=12)
plt.xlim(0.6, 1.0)
plt.ylabel('Model', fontsize=12)
plt.tight_layout()
plt.show()

## 4. SHAP Feature Importance Analysis
Visualizing overall feature contributions to model valuations:

In [ ]:
# Feature Importance ranking
with open('../backend/model/model_metadata.json', 'r') as f:
    metadata = json.load(f)

print("Best Model Selected:", metadata.get('best_model_name'))
print("Test R² Score:", metadata.get('best_metrics', {}).get('r2_score'))
print("Test RMSE:", f"₹{metadata.get('best_metrics', {}).get('rmse'):,}")